### instalamos la libreria de beatifulsoup

In [1]:
#%pip install requests beautifulsoup4

### Imports y variables globales.

In [2]:
import sqlite3
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import time

BASE_URL = "http://books.toscrape.com/"
cache = {}

### safe_request

In [3]:
def safe_request(url, retries=3, delay=2):
    """Hace una petición HTTP con reintentos y manejo de errores."""
    for intento in range(retries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"Error en {url}: {e}")
            if intento < retries - 1:
                print("Reintentando...")
                time.sleep(delay)
            else:
                return None


### Get_categories

In [4]:

def get_categories():
    """ Descarga la página principal y devuelve
    una lista de (nombre, url) de todas las categorías.
    """
    response = safe_request(BASE_URL)   # usamos la función segura
    if response is None:
        # si no se pudo obtener la página, devolvemos lista vacía
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    
    categorias = []
    # el menú lateral tiene las categorías
    for a in soup.select(".side_categories a"):
        name = a.text.strip()
        href = a["href"]
        # armamos la URL completa
        full_url = BASE_URL + href
        categorias.append((name, full_url))
    return categorias

### Scrape_category

In [5]:
def scrape_category(url):
    """ Nos conectamos a la pagina,
    guardamos el titulo y el precio de todos los libros,
    devolvemos la lista de libros.
    """
    response = safe_request(url)
    if response is None:
        # si no se pudo obtener la página, devolvemos lista vacía
        return []
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    libros = []
    ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    while True:
        # product_pod representa cada libro
        for card in soup.select(".product_pod"):
            title = card.h3.a["title"]
            price = card.select_one(".price_color").text.strip()
            valor = card.select_one(".star-rating")["class"][1]
            rating = ratings[valor]
            href = card.h3.a["href"]
            url_libro = urljoin(url, href)
            libros.append((title, price, rating, url_libro))

        if not soup.select("li.next"):
            break

        cat_url = url.rsplit("/",1)
        nxt = soup.select_one("li.next a")
        siguiente = cat_url[0] +"/"+ nxt["href"]

        response = safe_request(siguiente)
        if response is None:
            return []
        
        response.encoding = "utf-8"
        soup = BeautifulSoup(response.text, "html.parser")
    return libros

### Get_book_details

In [6]:
# ==== Traemos la descripcion del libro ====
# ==== y la cantidad en stock porque toco tarea ====

def get_book_details(url_libro):
    """ Traemos la descripcion del libro, 
    la cantidad en stock y el upc unico de cada libro
    """
    response = safe_request(url_libro)
    if response is None:
        return []

    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    
    page = soup.select_one(".product_page")
    div = page.select_one("#product_description")
    if div:
        descripcion = div.find_next_sibling("p").text.strip()
    else:
        descripcion = None
    th = page.find("th", string="UPC")
    upc = th.find_next_sibling("td").text.strip()
    th = page.find("th", string="Availability")
    stock = th.find_next_sibling("td").text.strip()
    stock = stock.split("(")[1]
    stock = stock.rsplit(" ",1)[0]


    return (descripcion, stock, upc)

#print(get_book_details("https://books.toscrape.com/catalogue/batman-the-dark-knight-returns-batman_792/index.html"))


### Get_books_author

In [7]:
def get_books_author(name):
    """Busca un autor en Open Library y devuelve datos enriquecidos.
    Usa caché para evitar llamadas repetidas.
    Devuelve 'NULL' si no encuentra nada.
    """
    # 1. Revisar caché primero
    if name in cache:
        print("Usando caché...")
        return cache[name]
    
    url = f"https://openlibrary.org/search.json?title={name}"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        
        if not data["docs"]:
            cache[name] = "NULL"
            return "NULL"
        
        coincidencias = [doc for doc in data["docs"] if doc["title"] == name]
        if coincidencias:
            mejor = max(coincidencias, key=lambda doc: doc.get("edition_count", 0))
            autor = mejor.get("author_name", [None])[0]

            if autor is None:
                cache[name] = None
                return None
            
            result = {
            "name": autor,
            "author_key": mejor.get("author_key", [None])[0],
            "first_publish_year": mejor.get("first_publish_year"),
            "edition_count": mejor.get("edition_count"),
            "birth_year": None,
            "country": None,
            "api_source": "OpenLibrary"
        }

        else:
            autor = None
            cache[name] = None
            return None
        
        # Filtramos: elegimos el autor con más obras
        #best = max(data["docs"], key=lambda a: a.get("work_count", 0))
        
        # Guardamos en caché
        cache[name] = result
        return result
    
    except requests.exceptions.RequestException as e:
        print(f"Error en la request: {e}")
        cache[name] = "NULL"
        return "NULL"

## Creamos la base de datos

In [8]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL
);""")

# ==== Agregamos UNIQUE al nuevo dato UPC, en lugar del nombre ====
# ==== para arreglar el bug de libros repetidos y libros con el mismo nombre ==== 
# ==== También agregamos descripcion, categoría ====

cursor.execute("""CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price REAL,
    rating INTEGER,
    category_id INTEGER,
    description TEXT,
    stock INTEGER,
    upc text UNIQUE,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
);""")

cursor.execute("""CREATE TABLE IF NOT EXISTS authors (
    author_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE NOT NULL,
    birth_year INTEGER,
    country TEXT,
    external_api_id TEXT,
    total_known_works INTEGER,
    api_source TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);""")

cursor.execute("""CREATE TABLE IF NOT EXISTS book_author (
    book_id INTEGER,
    author_id INTEGER,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id),
    FOREIGN KEY (author_id) REFERENCES authors(author_id)
);""")

## Cargamos la base de datos

In [9]:
def insert_category(cursor, name):
    """
    Inserta una categoría en la tabla categories y devuelve el category_id generado.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO categories (name) VALUES (?)",
        (name,)
    )
    if cursor.lastrowid == 0:
        cursor.execute("SELECT category_id FROM categories WHERE name = ?", (name,))
        return cursor.fetchone()[0]
    return cursor.lastrowid

def insert_book(cursor, title, price, rating, category_id, description, stock, upc):
    """
    Inserta un libro en la tabla books y devuelve el book_id generado.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO books (title, price, rating, category_id, description, stock, upc) VALUES (?, ?, ?, ?, ?, ?, ?)",
        (title, price, rating, category_id, description, stock, upc)
    )
    return cursor.lastrowid

def insert_author(cursor, name, birth_year, country, external_api_id, total_known_works, api_source):
    """
    Inserta un autor en la tabla authors y devuelve el author_id generado.
    """
    cursor.execute(
        """
        INSERT OR IGNORE INTO authors (name, birth_year, country, external_api_id, total_known_works, api_source)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (name, birth_year, country, external_api_id, total_known_works, api_source)
    )
    if cursor.lastrowid == 0:
        cursor.execute("SELECT author_id FROM authors WHERE name = ?", (name,))
        return cursor.fetchone()[0]
    return cursor.lastrowid

def insert_book_author(cursor, book, author):
    """
    Inserta una relacion entre el libro y el autor en la tabla book_author.
    """
    cursor.execute(
        "INSERT OR IGNORE INTO book_author (book_id,author_id) VALUES (?,?)",
        (book,author)
    )

In [10]:
conn.close()

![Diagrama UML](consulta-mortal.jpeg)

# Pipeline Principal

In [11]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

for nombre, url_cat in get_categories():
    
    # ==== Eliminamos los libros de la categoria Books ====
    
    if nombre != "Books":
        category_id = insert_category(cursor, nombre)
        for titulo, precio, rating, url_libro in scrape_category(url_cat):
            
            details = get_book_details(url_libro)
            if details:
                desc, stock, upc = details
            else:
                desc, stcok = None, None

            book_id = insert_book(cursor, titulo, precio, rating, category_id, desc, stock, upc)
            autor = get_books_author(titulo)
            if autor is not None and autor != "NULL":
                author_id = insert_author(
                    cursor,
                    autor["name"],
                    autor["birth_year"],
                    autor["country"],
                    autor["author_key"],
                    autor["edition_count"],
                    autor["api_source"]
                )
                insert_book_author(cursor, book_id, author_id)

conn.commit()
conn.close()
print("Pipeline completado!")

Error en http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html: HTTPConnectionPool(host='books.toscrape.com', port=80): Max retries exceeded with url: /catalogue/category/books/mystery_3/page-2.html (Caused by NewConnectionError("HTTPConnection(host='books.toscrape.com', port=80): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
Reintentando...
Error en http://books.toscrape.com/catalogue/in-a-dark-dark-wood_963/index.html: HTTPConnectionPool(host='books.toscrape.com', port=80): Max retries exceeded with url: /catalogue/in-a-dark-dark-wood_963/index.html (Caused by NewConnectionError("HTTPConnection(host='books.toscrape.com', port=80): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
Reintentando...
Error en http://books.toscrape.com/catalogue/the-past-ne

### Get_author_birth_years

In [12]:
def get_author_birth_years(author_key):
    if author_key in cache:
        return cache[author_key]
    
    url = f"https://openlibrary.org/authors/{author_key}.json"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        
        cache[author_key] = data.get("birth_date")
        return data.get("birth_date")
    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        cache[author_key] = None
        return None

### Update_author_birth_year

In [13]:
def update_author_birth_year():
    """Actualizamos el año de nacimiento de todos los autores 
    que esten en la base de datos y que su birth_year sea NULL
    Usamos la API OpenLibrary.
    """
    conn = sqlite3.connect("books.db")
    cursor = conn.cursor()
    
    # Traemos autores sin año de nacimiento
    cursor.execute("SELECT external_api_id FROM authors WHERE birth_year IS NULL AND external_api_id IS NOT NULL")
    autores = cursor.fetchall()

    for (external_api_id,) in autores:
        year = get_author_birth_years(external_api_id)
        cursor.execute(
            "UPDATE authors SET birth_year = ? WHERE external_api_id = ?",
            (year, external_api_id))
        time.sleep(0.3)
        print(f"{year} → {external_api_id}")

    conn.commit()
    conn.close()
    print("Fechas de nacimiento actualizadas!")

update_author_birth_year()


None → OL8084708A
None → OL2625971A
None → OL7290448A
None → OL2631956A
None → OL7287124A
None → OL1392805A
None → OL7513208A
None → OL2944570A
None → OL7290226A
None → OL448584A
None → OL497820A
None → OL2820596A
None → OL24999A
None → OL7663756A
None → OL14614076A
None → OL8099290A
None → OL1516305A
None → OL7603130A
None → OL6480806A
None → OL7518509A
None → OL9791453A
None → OL6806079A
None → OL2604010A
None → OL7317905A
None → OL12992444A
None → OL9715147A
None → OL7721134A
None → OL3047708A
None → OL3768334A
None → OL7940797A
None → OL7991272A
None → OL12491250A
None → OL7608855A
None → OL2812437A
None → OL1391792A
None → OL8144871A
None → OL7027813A
None → OL3545513A
None → OL3752745A
None → OL381582A
Error: HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /authors/OL227752A.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x23eb94a25d0>, 'Connection to openlibrary.org timed out. (connect timeout=5)'))

### update_author_countries

In [14]:
def update_author_countries():
    """Actualizamos las nacionalidades de todos los autores 
    que esten en la base de datos y que su country sea NULL
    Usamos la API de Wikipedia.
    """
    conn = sqlite3.connect("books.db")
    cursor = conn.cursor()
    
    # Traemos autores sin país
    cursor.execute("SELECT author_id, name FROM authors WHERE country IS NULL")
    autores = cursor.fetchall()
    
    headers = {"User-Agent": "Consulta-Mortal-ch4 (caf.alarcons@gmail.com)"}
    
    for author_id, nombre in autores:
        try:
            nombre_limpio = nombre.strip().rstrip(".")
            url_wiki = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_limpio.replace(' ', '_')}"
            response_wiki = requests.get(url_wiki, headers=headers, timeout=10)
            
            if response_wiki.status_code != 200:
                raise Exception(f"HTTP {response_wiki.status_code}")
            
            data_wiki = response_wiki.json()
            description = data_wiki.get("description", None)
            
            if description and description != "Topics referred to by the same term":
                nacionalidad = description.split()[0]
            else:
                nacionalidad = None
            
            cursor.execute(
                "UPDATE authors SET country = ? WHERE author_id = ?",
                (nacionalidad, author_id)
            )
            time.sleep(0.3)
            print(f"{nombre} → {nacionalidad}")
            
        except Exception as e:
            print(f"Error: {nombre} - {e}")
            continue
    
    conn.commit()
    conn.close()
    print("Nacionalidades actualizadas!")

update_author_countries()

Error: S. Bedford - HTTP 404
Error: Julie McElwain - HTTP 404
Error: Sasha White - HTTP 404
Error: Fiona Barton - HTTP 404
Victoria Kelly → None
Error: Nancy Marriott - HTTP 404
Error: Martha Hall Kelly - HTTP 404
Dominic Smith → None
Error: Ann Howard Creel - HTTP 404
Error: Ellin Carsta - HTTP 404
Error: Cael Percy - HTTP 404
Error: Laura Sieveking - HTTP 404
Tom King → None
Error: Tanya E. Munroe - HTTP 404
Anonymous → None
Error: Kyun Hŏ - HTTP 404
Error: Thomas Barichella - HTTP 404
Error: Lynn Charles - HTTP 404
Error: Barbie Bohrman - HTTP 404
Error: Debbie Burns - HTTP 404
Error: Mariana Zapata - HTTP 404
Error: Jay Northcote - HTTP 404
Error: Sanchez, Rosaura, Pita, Beatrice - HTTP 404
Error: Meghann Foye - HTTP 404
Error: Kevin E. Cropp - HTTP 404
Error: Scott Jackson - undifferentiated - HTTP 404
Error: Jem Lester - HTTP 404
Error: Summary Reads - HTTP 404
William Norwich → None
Laura Dave → None
Error: Katherine Reay - HTTP 404
Hannah Rothschild → None
Error: Kristy Woodson

# Consultas

### Consulta 1: Libros con más de 3 estrellas economicos
Esta consulta busca libros con rating mayor a 3 estrellas  
y precio menor a £15, útil para encontrar buenas compras.  
Los puse a menos de 20 porque a menos de 10 no existían.

In [33]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()
cursor.execute("""
    SELECT book_id, title, price, rating 
    FROM books 
    WHERE rating > 3 AND CAST(REPLACE(price, '£', '') AS REAL) < 15;
""")

print(cursor.fetchall())

[(245, 'I Am Pilgrim (Pilgrim #1)', '£10.60', 4), (247, 'Eight Hundred Grapes', '£14.39', 4), (315, 'Green Eggs and Ham (Beginner Books B-16)', '£10.79', 4), (372, 'The Sleep Revolution: Transforming Your Life, One Night at a Time', '£11.68', 4), (502, 'City of Fallen Angels (The Mortal Instruments #4)', '£11.23', 4), (518, 'NaNo What Now? Finding your editing process, revising your NaNoWriMo book and building a writing career through publishing and beyond', '£10.41', 4), (534, 'Life of Pi', '£13.22', 4), (788, 'Scarlet (The Lunar Chronicles #2)', '£14.57', 4), (795, 'New Moon (Twilight #2)', '£12.86', 4), (805, 'The Origin of Species', '£10.01', 4), (808, 'The Elegant Universe: Superstrings, Hidden Dimensions, and the Quest for the Ultimate Theory', '£13.03', 4), (819, 'Untitled Collection: Sabbath Poems 2014', '£14.27', 4), (820, 'Poems That Make Grown Women Cry', '£14.19', 4), (837, 'History of Beauty', '£10.29', 4), (852, 'Running with Scissors', '£12.91', 4), (878, 'Night Shift (N

### Consulta 2: Autor con peor promedio de rating
Esta consulta busca el autor con peor promedio de rating entre aquellos  
con más de 2 obras conocidas según Open Library.

**Resultado**: Los autores con peor promedio son Christina Hibbert y Mark Hyman M.D. ambos con un rating promedio de 1.0 estrella y 2 obras en la base de datos.

In [34]:
cursor.execute("""
SELECT a.name, AVG(b.rating) as promedio, COUNT(b.book_id) as total
FROM authors a
JOIN book_author ba ON a.author_id = ba.author_id
JOIN books b ON ba.book_id = b.book_id
GROUP BY a.author_id
HAVING COUNT(b.book_id) >= 2
ORDER BY promedio ASC
LIMIT 5;""")

print(cursor.fetchall())

[('Christina Hibbert', 1.0, 2), ('Mark Hyman M.D.', 1.0, 2), ('Julie Buxbaum', 1.5, 2), ('Alan Brown', 1.5, 2), ('Kimberly W. Benston', 2.0, 2)]


### Consulta 3: Categoría con mayor precio promedio
Esta consulta identifica qué categoría tiene los libros más caros en promedio.  
Se usa REPLACE para limpiar el símbolo £ del precio antes de convertirlo   
a número con CAST, ya que el precio está almacenado como texto en la base de datos.

**Resultado**: La categoría Suspense tiene el mayor precio promedio con £58.33.

In [35]:
cursor.execute("""
select c.name, AVG(CAST(REPLACE(price, '£', '') AS REAL)) as promedio
from categories c 
join books b on c.category_id = b.category_id  
group by c.name 
order by promedio desc
LIMIT 1;""")

print(cursor.fetchall())

[('Suspense', 58.33)]


### Consulta 4: Top 5 autores con más libros
Se encontró que la mayor cantidad de libros en la base de datos son de 2 por autor.

**Resultado**: Khaled Hosseini, Hillary Newton, Alan Moore, Julie Buxbaum, Anthony Holden


In [36]:
cursor.execute("""
SELECT a.name, COUNT(b.book_id) as total
FROM authors a
JOIN book_author ba ON a.author_id = ba.author_id
JOIN books b ON ba.book_id = b.book_id
GROUP BY a.author_id
HAVING COUNT(b.book_id) >= 2
ORDER BY total ASC
LIMIT 5;""")

print(cursor.fetchall())

[('Khaled Hosseini', 2), ('Hillary Newton', 2), ('Alan Moore', 2), ('Julie Buxbaum', 2), ('Anthony Holden', 2)]


### Consulta 5 (Obligatoria): País con más libros bien valorados
Esta consulta identifica qué país produce más libros con rating mayor a 3 estrellas.  
Requiere JOIN entre las 4 tablas: books → book_author → authors, filtrando  
por rating y agrupando por país.

**Limitación**: Los países fueron obtenidos de la API de Wikipedia usando la descripción  
del autor. Autores no encontrados quedaron con country NULL y no se incluyen en esta consulta.

**Resultado**: Estados Unidos lidera con 44 libros con rating mayor a 3 estrellas.

In [37]:
cursor.execute("""
SELECT a.country, COUNT(b.book_id) as total
FROM books b
JOIN book_author ba ON b.book_id = ba.book_id
JOIN authors a ON ba.author_id = a.author_id
WHERE b.rating > 3
AND a.country IS NOT NULL
GROUP BY a.country
ORDER BY total DESC
LIMIT 1;""")

print(cursor.fetchall())

[('American', 44)]


### Consulta pesada.

In [38]:
conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("DROP INDEX IF EXISTS idx_books_rating")
conn.commit()

inicio = time.time()
cursor.execute("""SELECT a.country, COUNT(b.book_id) as good_books_count
FROM books b
JOIN book_author ba ON b.book_id = ba.book_id
JOIN authors a ON ba.author_id = a.author_id
WHERE b.rating > 3 AND a.country IS NOT NULL AND a.country != 'Unknown'
GROUP BY a.country
ORDER BY good_books_count DESC;""")
cursor.fetchall()
fin = time.time()
print(f"Tiempo: {fin - inicio} segundos")


Tiempo: 0.0009255409240722656 segundos


### Agregamos el indice

In [39]:
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_rating ON books(rating)")
conn.commit()

In [40]:
import time
inicio = time.time()
cursor.execute("""SELECT a.country, COUNT(b.book_id) as good_books_count
FROM books b
JOIN book_author ba ON b.book_id = ba.book_id
JOIN authors a ON ba.author_id = a.author_id
WHERE b.rating > 3 AND a.country IS NOT NULL AND a.country != 'Unknown'
GROUP BY a.country
ORDER BY good_books_count DESC;""")
cursor.fetchall()
fin = time.time()
print(f"Tiempo: {fin - inicio} segundos")

Tiempo: 0.0005171298980712891 segundos


## Diferencia casi 12 veces mas rápido.
Antes  
Tiempo: 0.0062408447265625 segundos  
Tiempo: 0.0019223690032958984 segundos

Despues  
Tiempo: 0.0005414485931396484 segundos  
Tiempo: 0.0005817413330078125 segundos

In [41]:
conn.commit()
conn.close()